# Build InfoCuria case skeleton

This notebook builds the **case skeleton** for the Court/InfoCuria part of the project.

The aim is to create a stable, case-level spine that every later pipeline stage can join back to. The downstream questions are:

1. **Completeness** — did we find the case at all, and does later metadata/document scraping return documents for it?
2. **Relevance** — does the case contain at least one relevant document type, such as a judgment, order, Advocate General opinion, summary, or decision?
3. **Downloadability** — for relevant documents, did we actually download and convert a readable copy?

This notebook only handles the first part: building the initial InfoCuria case skeleton from the `affair` tab. It does **not** classify documents and does **not** download documents.

It can scrape InfoCuria from scratch, update an existing raw scrape cache, or rebuild the CSV/Excel/JSONL manifests from saved raw JSONL without scraping again. It also adds normalized case keys, including `case_suffix` and `internal_key`.

## Why six scrape passes?

InfoCuria search results are not perfectly stable across all query/sort strategies. Some cases appear in one pass and disappear in another, for reasons that are not always transparent from the API response. To maximize recall, this notebook therefore combines six independent passes:

1. `introductionDate` ascending
2. `introductionDate` descending
3. `closeDate` ascending
4. `closeDate` descending
5. case-number wildcard buckets ascending, for example `C-*/00`, `T-*/00`
6. case-number wildcard buckets descending, for example `C-*/00`, `T-*/00`

The final case skeleton is deduplicated after all six passes. The audit manifest keeps the provenance columns so that dropped, duplicated, or pass-specific cases can still be inspected.

## Run modes

Set `SCRAPE_MODE` in the configuration cell:

```python
SCRAPE_MODE = "rebuild_from_cache"  # no scraping; rebuild CSV/Excel/JSONL manifests from saved raw JSONL
SCRAPE_MODE = "fresh_scrape"        # scrape from zero and overwrite everything
SCRAPE_MODE = "update_scrape"       # load existing raw JSONL, scrape again, append/update, then rebuild outputs
```

Use:

* `"fresh_scrape"` when you want to fully rebuild the case universe from zero.
* `"update_scrape"` when you want to preserve the existing raw JSONL cache, scrape again, and append/update rows.
* `"rebuild_from_cache"` when you only want to regenerate the CSV, JSONL, and Excel outputs from already saved raw scrape data.

## Scope and deliberate exclusions

The project focuses on Court of Justice and General Court cases, so this notebook keeps `C` and `T` case prefixes. Other prefixes are deliberately excluded from the final skeleton. For example, `Avis` / opinion-style material and other non-`C`/`T` rows can appear in broader InfoCuria searches, but they are not part of this case spine. Similarly, `F` cases are excluded because they are Civil Service Tribunal cases and are outside the current project scope.

## Normalized case keys

The notebook extracts and stores normalized case-number components, including:

* `case_prefix`, for example `C` or `T`
* `case_number`, for example `1`
* `case_number_padded`, for example `0001`
* `case_year_2digit`, for example `00`
* `case_year_full`, for example `2000`
* `case_suffix`, for example `SA`, `P`, `PPU`, `DEP`, or empty
* `internal_key`, for example `2000/0001/C/SA/`

The `internal_key` is intended as a stable normalized case-level join key.

## Outputs

CSV outputs are written under:

```text
output/court_infocuria/
```

JSONL outputs are written under:

```text
output/court_infocuria/jsonl/
```

Excel outputs are written under:

```text
output/court_infocuria/excel/
```

The main CSV outputs are:

```text
infocuria_case_scrape_manifest.csv
infocuria_case_scrape_audit_manifest.csv
```

The regular scrape manifest is the slim file intended for downstream use. The audit manifest keeps the larger set of provenance/debug columns.

The JSONL outputs preserve the raw and reconstructed scrape data in a more robust format for later rebuilding. The Excel outputs are convenience files for manual inspection.



## Modified deduplication workflow

This version does **not** rerun the InfoCuria scrape.

It uses:

```python
SCRAPE_MODE = "rebuild_from_cache"
```

and reloads the existing raw JSONL cache.

Final deduplication now prioritizes:

```text
procedureId → affId → publishedId → publishedAffId → fallback identifiers
```

This preserves separate procedure records when the same `affId` appears with more than one distinct `procedureId`.


In [13]:
import os
import re
import json
import time
import random
import logging
from pathlib import Path
from datetime import datetime, date, timedelta

import requests
import pandas as pd
from tqdm.auto import tqdm

In [14]:
# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
OUTPUT_DIR = PROJECT_ROOT / "output"
LOGS_DIR = PROJECT_ROOT / "logs"

COURT_INFOCURIA_RAW_DIR = RAW_DATA_DIR / "court_infocuria"
COURT_INFOCURIA_JSON_DIR = COURT_INFOCURIA_RAW_DIR / "api_responses_final"
COURT_INFOCURIA_OUTPUT_DIR = OUTPUT_DIR / "court_infocuria"
COURT_INFOCURIA_JSONL_OUTPUT_DIR = COURT_INFOCURIA_OUTPUT_DIR / "jsonl"
COURT_INFOCURIA_EXCEL_OUTPUT_DIR = COURT_INFOCURIA_OUTPUT_DIR / "excel"
COURT_INFOCURIA_LOGS_DIR = LOGS_DIR / "court_infocuria"

for path in [
    COURT_INFOCURIA_RAW_DIR,
    COURT_INFOCURIA_JSON_DIR,
    COURT_INFOCURIA_OUTPUT_DIR,
    COURT_INFOCURIA_JSONL_OUTPUT_DIR,
    COURT_INFOCURIA_EXCEL_OUTPUT_DIR,
    COURT_INFOCURIA_LOGS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Output:", COURT_INFOCURIA_OUTPUT_DIR)
print("JSONL output:", COURT_INFOCURIA_JSONL_OUTPUT_DIR)
print("Excel output:", COURT_INFOCURIA_EXCEL_OUTPUT_DIR)
print("Logs:", COURT_INFOCURIA_LOGS_DIR)


NOTEBOOK_DIR: /home/edik/projects/eccjeu/code/notebooks
PROJECT_ROOT: /home/edik/projects/eccjeu
Output: /home/edik/projects/eccjeu/output/court_infocuria
JSONL output: /home/edik/projects/eccjeu/output/court_infocuria/jsonl
Excel output: /home/edik/projects/eccjeu/output/court_infocuria/excel
Logs: /home/edik/projects/eccjeu/logs/court_infocuria


In [15]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

SEARCH_URL = "https://infocuriaws.curia.europa.eu/elastic-connector/search"

HEADERS = {
    "accept": "application/json",
    "content-type": "application/json; charset=UTF-8",
    "origin": "https://infocuria.curia.europa.eu",
    "referer": "https://infocuria.curia.europa.eu/",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome Safari",
}

START_YEAR = 1950
END_YEAR = 2028
END_YEAR_EXCLUSIVE = END_YEAR + 1

KEEP_PREFIXES = {"C", "T"}
WILDCARD_PREFIXES = ["C", "T"]

PAGE_SIZE = 100
REQUEST_TIMEOUT_SECONDS = 90

SLEEP_BETWEEN_PAGES_SECONDS = 1.0
SLEEP_BETWEEN_WINDOWS_SECONDS = 1.5
SLEEP_BETWEEN_BUCKETS_SECONDS = 1.0
JITTER_SECONDS = 0.7

MAX_HITS_PER_WINDOW = 1500
MIN_WINDOW_DAYS = 1
DEEP_PAGINATION_LIMIT = 10000
DEEP_PAGINATION_WARNING_AT = 9500

CHECKPOINT_EVERY_PASS_OR_BUCKETS = 10
STOP_AFTER_CONSECUTIVE_EMPTY_PAGES = 2
SAVE_RAW_JSON = False

# ---------------------------------------------------------------------
# v3 run mode
# ---------------------------------------------------------------------
# Options:
#   "fresh_scrape"        = scrape from zero and overwrite raw JSONL + manifests
#   "update_scrape"       = load existing raw JSONL, scrape again, append/update, rebuild manifests
#   "rebuild_from_cache"  = no scraping; rebuild manifests from saved raw JSONL/Excel/CSV
SCRAPE_MODE = "rebuild_from_cache"
VALID_SCRAPE_MODES = {"fresh_scrape", "update_scrape", "rebuild_from_cache"}
if SCRAPE_MODE not in VALID_SCRAPE_MODES:
    raise ValueError(f"SCRAPE_MODE must be one of {sorted(VALID_SCRAPE_MODES)}, got {SCRAPE_MODE!r}")

WRITE_CSV = True
WRITE_JSONL = True
WRITE_EXCEL = True

SCRAPE_MANIFEST_CSV = COURT_INFOCURIA_OUTPUT_DIR / "infocuria_case_scrape_manifest.csv"
AUDIT_MANIFEST_CSV = COURT_INFOCURIA_OUTPUT_DIR / "infocuria_case_scrape_audit_manifest.csv"

RAW_CASES_JSONL = COURT_INFOCURIA_JSONL_OUTPUT_DIR / "infocuria_case_scrape_raw_cases.jsonl"
SCRAPE_MANIFEST_JSONL = COURT_INFOCURIA_JSONL_OUTPUT_DIR / "infocuria_case_scrape_manifest.jsonl"
AUDIT_MANIFEST_JSONL = COURT_INFOCURIA_JSONL_OUTPUT_DIR / "infocuria_case_scrape_audit_manifest.jsonl"

SCRAPE_MANIFEST_XLSX = COURT_INFOCURIA_EXCEL_OUTPUT_DIR / "infocuria_case_scrape_manifest.xlsx"
AUDIT_MANIFEST_XLSX = COURT_INFOCURIA_EXCEL_OUTPUT_DIR / "infocuria_case_scrape_audit_manifest.xlsx"

DATE_PASSES = [
    {"pass_name": "introductionDate_ASC", "kind": "date", "filter_field": "introductionDate", "sort_term": "INTRODUCTION_DATE", "sort_direction": "ASC"},
    {"pass_name": "introductionDate_DESC", "kind": "date", "filter_field": "introductionDate", "sort_term": "INTRODUCTION_DATE", "sort_direction": "DESC"},
    {"pass_name": "closeDate_ASC", "kind": "date", "filter_field": "closeDate", "sort_term": "CLOSE_DATE", "sort_direction": "ASC"},
    {"pass_name": "closeDate_DESC", "kind": "date", "filter_field": "closeDate", "sort_term": "CLOSE_DATE", "sort_direction": "DESC"},
]

WILDCARD_PASSES = [
    {"pass_name": "case_number_wildcard_ASC", "kind": "wildcard", "sort_term": "AFF_NUM", "sort_direction": "ASC"},
    {"pass_name": "case_number_wildcard_DESC", "kind": "wildcard", "sort_term": "AFF_NUM", "sort_direction": "DESC"},
]

SCRAPE_PASSES = DATE_PASSES + WILDCARD_PASSES

print("SCRAPE_MODE:", SCRAPE_MODE)
print("Slim scrape manifest CSV:", SCRAPE_MANIFEST_CSV)
print("Audit scrape manifest CSV:", AUDIT_MANIFEST_CSV)
print("Raw cases JSONL:", RAW_CASES_JSONL)
print("Excel folder:", COURT_INFOCURIA_EXCEL_OUTPUT_DIR)
print("Excel audit manifest fallback:", AUDIT_MANIFEST_XLSX)
print("Excel slim manifest fallback:", SCRAPE_MANIFEST_XLSX)


SCRAPE_MODE: rebuild_from_cache
Slim scrape manifest CSV: /home/edik/projects/eccjeu/output/court_infocuria/infocuria_case_scrape_manifest.csv
Audit scrape manifest CSV: /home/edik/projects/eccjeu/output/court_infocuria/infocuria_case_scrape_audit_manifest.csv
Raw cases JSONL: /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_case_scrape_raw_cases.jsonl
Excel folder: /home/edik/projects/eccjeu/output/court_infocuria/excel
Excel audit manifest fallback: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_case_scrape_audit_manifest.xlsx
Excel slim manifest fallback: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_case_scrape_manifest.xlsx


In [16]:
# ---------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = COURT_INFOCURIA_LOGS_DIR / f"scrape_court_infocuria_final_{timestamp}.log"

logger = logging.getLogger("scrape_court_infocuria_final")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(log_path, encoding="utf-8")
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(stream_handler)

print("Log file:", log_path)

Log file: /home/edik/projects/eccjeu/logs/court_infocuria/scrape_court_infocuria_final_20260715_142241.log


In [17]:
# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def polite_sleep(base_seconds: float):
    if base_seconds and base_seconds > 0:
        time.sleep(base_seconds + random.uniform(0, JITTER_SECONDS))


def parse_date_yyyy_mm_dd(s: str) -> date:
    return datetime.strptime(s, "%Y-%m-%d").date()


def fmt_date(d: date) -> str:
    return d.strftime("%Y-%m-%d")


def window_days(start_date: str, end_date: str) -> int:
    return (parse_date_yyyy_mm_dd(end_date) - parse_date_yyyy_mm_dd(start_date)).days


def split_date_window(start_date: str, end_date: str):
    s = parse_date_yyyy_mm_dd(start_date)
    e = parse_date_yyyy_mm_dd(end_date)
    days = (e - s).days
    if days <= 1:
        return None
    mid = s + timedelta(days=days // 2)
    return (fmt_date(s), fmt_date(mid)), (fmt_date(mid), fmt_date(e))


def yy_from_year(year: int) -> str:
    return f"{year % 100:02d}"


def bucket_search_term(prefix: str, year: int) -> str:
    return f"{prefix}-*/{yy_from_year(year)}"


def get_ml_value(ml_list, lang="en"):
    if not isinstance(ml_list, list):
        return None
    for item in ml_list:
        if isinstance(item, dict) and lang in item:
            return item[lang]
    return None


def extract_ml_labels(items, lang="en"):
    if not isinstance(items, list):
        return None
    labels = []
    for item in items:
        if not isinstance(item, dict):
            continue
        code = item.get("code")
        label = get_ml_value(item.get("label", []), lang)
        if code and label:
            labels.append(f"{code}: {label}")
        elif code:
            labels.append(str(code))
        elif label:
            labels.append(str(label))
    return " | ".join(labels) if labels else None


def first_nonempty(*values):
    for v in values:
        if v is not None and not (isinstance(v, float) and pd.isna(v)) and str(v).strip() != "":
            return v
    return None

In [18]:
# ---------------------------------------------------------------------
# Clean case-number extraction
# ---------------------------------------------------------------------

CASE_NUMBER_RE = re.compile(
    r"""
    (?P<prefix>[A-Z]{1,4})
    \s*[-–—]?\s*
    (?P<number>\d{1,4})
    \s*[/\-]\s*
    (?P<year>\d{2,4})
    (?P<suffix>(?:\s+[A-Z][A-Z0-9]*(?:[-_/][A-Z0-9]+)*)*)
    """,
    re.VERBOSE,
)


def normalize_year(year_text):
    if year_text is None:
        return None
    y = str(year_text).strip()
    if not y.isdigit():
        return None
    if len(y) == 4:
        return int(y)
    if len(y) == 2:
        yy = int(y)
        return 1900 + yy if yy >= 50 else 2000 + yy
    return None


def normalize_suffix(suffix_text):
    """Return a compact case suffix such as P, P-DEP, SA, PPU, R, or empty string."""
    if suffix_text is None:
        return ""
    suffix = str(suffix_text).strip().upper()
    suffix = suffix.replace("–", "-").replace("—", "-")
    suffix = re.sub(r"\s+", "-", suffix)
    suffix = suffix.strip("-/")
    return suffix


def make_internal_key(year_full, case_number_padded, case_prefix, case_suffix=""):
    """Build YYYY/NNNN/PREFIX/SUFFIX/; the suffix slot is kept even when suffix is empty."""
    if year_full is None or pd.isna(year_full):
        return None
    if case_number_padded is None or pd.isna(case_number_padded) or str(case_number_padded).strip() == "":
        return None
    if case_prefix is None or pd.isna(case_prefix) or str(case_prefix).strip() == "":
        return None
    year = int(float(year_full))
    number = str(case_number_padded).strip().zfill(4)
    prefix = str(case_prefix).strip().upper()
    suffix = normalize_suffix(case_suffix)
    return f"{year:04d}/{number}/{prefix}/{suffix}/"


def clean_case_number(raw):
    text = "" if raw is None or (isinstance(raw, float) and pd.isna(raw)) else str(raw).strip()
    if not text:
        return {
            "case_number_raw": None,
            "case_number_clean": None,
            "case_prefix": None,
            "case_suffix": "",
            "case_number_int": None,
            "case_number_padded": None,
            "case_year_raw": None,
            "case_year_full": None,
            "internal_key": None,
            "case_number_clean_status": "missing_raw_case_number",
        }

    text_norm = text.upper().replace("–", "-").replace("—", "-")
    match = CASE_NUMBER_RE.search(text_norm)
    if not match:
        return {
            "case_number_raw": text,
            "case_number_clean": None,
            "case_prefix": None,
            "case_suffix": "",
            "case_number_int": None,
            "case_number_padded": None,
            "case_year_raw": None,
            "case_year_full": None,
            "internal_key": None,
            "case_number_clean_status": "no_case_number_pattern",
        }

    prefix = match.group("prefix")
    number_int = int(match.group("number"))
    number_padded = f"{number_int:04d}"
    year_raw = match.group("year")
    year_full = normalize_year(year_raw)
    year_2 = f"{year_full % 100:02d}" if year_full is not None else year_raw[-2:]
    case_suffix = normalize_suffix(match.group("suffix"))
    clean = f"{prefix}-{number_int}/{year_2}" + (f" {case_suffix}" if case_suffix else "")
    internal_key = make_internal_key(year_full, number_padded, prefix, case_suffix)

    return {
        "case_number_raw": text,
        "case_number_clean": clean,
        "case_prefix": prefix,
        "case_suffix": case_suffix,
        "case_number_int": number_int,
        "case_number_padded": number_padded,
        "case_year_raw": year_raw,
        "case_year_full": year_full,
        "internal_key": internal_key,
        "case_number_clean_status": "ok",
    }


def add_case_key_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add/refresh case_prefix, case_suffix, padded number, year, and internal_key columns from case_number_raw."""
    if df.empty:
        return df.copy()
    out = df.copy()
    source_col = "case_number_raw" if "case_number_raw" in out.columns else None
    if source_col is None:
        for candidate in ["publishedId", "publishedAffId", "affId", "case_number_clean"]:
            if candidate in out.columns:
                source_col = candidate
                break
    if source_col is None:
        raise KeyError("Cannot add case key columns: no case-number-like column found.")

    parsed = out[source_col].apply(clean_case_number).apply(pd.Series)
    for col in parsed.columns:
        if col == "case_number_raw" and "case_number_raw" in out.columns:
            continue
        out[col] = parsed[col]
    if "case_number_raw" not in out.columns:
        out["case_number_raw"] = parsed["case_number_raw"]
    return out


def is_relevant_prefix(row_or_dict) -> bool:
    prefix = row_or_dict.get("case_prefix")
    return prefix in KEEP_PREFIXES


In [19]:
# ---------------------------------------------------------------------
# InfoCuria API calls
# ---------------------------------------------------------------------

def make_date_payload(page_number=0, page_size=PAGE_SIZE, start_date=None, end_date=None, filter_field="introductionDate", sort_term="INTRODUCTION_DATE", sort_direction="ASC"):
    from_ = page_number * page_size + 1
    to_ = from_ + page_size - 1
    payload = {
        "searchTerm": "",
        "multiSearchTerms": [],
        "sortTermList": [{"sortDirection": sort_direction, "sortTerm": sort_term, "sortSourceTab": "affair"}],
        "pagination": {"pageNumber": page_number, "pageSize": page_size, "from": from_, "to": to_},
        "language": "EN",
        "tabName": "affair",
        "isAllTabsRequest": False,
        "ecli": "",
        "publishedId": "",
        "usualName": "",
        "logicDocId": "",
        "isSearchExact": True,
        "searchSources": ["document", "metadata"],
    }
    if start_date and end_date and filter_field:
        payload["filtersValue"] = [{"field": filter_field, "values": [start_date, end_date], "valuesWithFullHierarchy": [filter_field]}]
    return payload


def make_wildcard_payload(search_term: str, page_number=0, page_size=PAGE_SIZE, sort_direction="ASC"):
    from_ = page_number * page_size + 1
    to_ = from_ + page_size - 1
    return {
        "searchTerm": search_term,
        "multiSearchTerms": [],
        "sortTermList": [{"sortDirection": sort_direction, "sortTerm": "AFF_NUM", "sortSourceTab": "affair"}],
        "pagination": {"pageNumber": page_number, "pageSize": page_size, "from": from_, "to": to_},
        "language": "EN",
        "tabName": "affair",
        "isAllTabsRequest": True,
        "ecli": "",
        "publishedId": "",
        "usualName": "",
        "logicDocId": "",
        "isSearchExact": True,
        "searchSources": ["document", "metadata"],
    }


def post_json(payload, max_retries=6):
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(SEARCH_URL, headers=HEADERS, json=payload, timeout=REQUEST_TIMEOUT_SECONDS)
            if response.ok:
                return response.json()
            last_error = f"HTTP {response.status_code}: {response.text[:500]}"
        except Exception as exc:
            last_error = repr(exc)

        wait = min(90, 2 ** attempt) + random.uniform(0, 1.5)
        logger.warning("Request failed attempt %s/%s: %s | sleeping %.1fs", attempt, max_retries, last_error, wait)
        time.sleep(wait)

    raise RuntimeError(f"InfoCuria request failed after {max_retries} retries: {last_error}")


def fetch_date_page(page_number=0, page_size=PAGE_SIZE, **kwargs):
    return post_json(make_date_payload(page_number=page_number, page_size=page_size, **kwargs))


def fetch_wildcard_page(search_term: str, page_number=0, page_size=PAGE_SIZE, sort_direction="ASC"):
    return post_json(make_wildcard_payload(search_term=search_term, page_number=page_number, page_size=page_size, sort_direction=sort_direction))

print("Date payload sample:")
print(json.dumps(make_date_payload(page_number=0, page_size=20, start_date="2025-01-01", end_date="2026-01-01"), indent=2, ensure_ascii=False))
print("\nWildcard payload sample:")
print(json.dumps(make_wildcard_payload("C-*/25", page_number=0, page_size=20), indent=2, ensure_ascii=False))

Date payload sample:
{
  "searchTerm": "",
  "multiSearchTerms": [],
  "sortTermList": [
    {
      "sortDirection": "ASC",
      "sortTerm": "INTRODUCTION_DATE",
      "sortSourceTab": "affair"
    }
  ],
  "pagination": {
    "pageNumber": 0,
    "pageSize": 20,
    "from": 1,
    "to": 20
  },
  "language": "EN",
  "tabName": "affair",
  "isAllTabsRequest": false,
  "ecli": "",
  "publishedId": "",
  "usualName": "",
  "logicDocId": "",
  "isSearchExact": true,
  "searchSources": [
    "document",
    "metadata"
  ],
  "filtersValue": [
    {
      "field": "introductionDate",
      "values": [
        "2025-01-01",
        "2026-01-01"
      ],
      "valuesWithFullHierarchy": [
        "introductionDate"
      ]
    }
  ]
}

Wildcard payload sample:
{
  "searchTerm": "C-*/25",
  "multiSearchTerms": [],
  "sortTermList": [
    {
      "sortDirection": "ASC",
      "sortTerm": "AFF_NUM",
      "sortSourceTab": "affair"
    }
  ],
  "pagination": {
    "pageNumber": 0,
    "pageSize

In [20]:
# ---------------------------------------------------------------------
# Flatten search hits
# ---------------------------------------------------------------------

def flatten_case_hit(hit, scrape_pass=None, scrape_kind=None, scrape_window_start=None, scrape_window_end=None, page_number=None, position_on_page=None, bucket_prefix=None, bucket_year=None, bucket_search_term=None, sort_direction=None):
    c = hit.get("content", hit)

    raw_case = first_nonempty(c.get("publishedId"), c.get("publishedAffId"), c.get("affaire"), c.get("caseNumber"))
    clean = clean_case_number(raw_case)

    row = {
        **clean,
        "scrape_pass": scrape_pass,
        "scrape_kind": scrape_kind,
        "scrape_window_start": scrape_window_start,
        "scrape_window_end": scrape_window_end,
        "bucket_prefix": bucket_prefix,
        "bucket_year": bucket_year,
        "bucket_year_yy": yy_from_year(int(bucket_year)) if bucket_year is not None else None,
        "bucket_search_term": bucket_search_term,
        "scrape_sort_direction": sort_direction,
        "scrape_page_number": page_number,
        "scrape_position_on_page": position_on_page,
        "affId": c.get("affId"),
        "publishedId": c.get("publishedId"),
        "publishedAffId": c.get("publishedAffId"),
        "procedureId": c.get("procedureId"),
        "case_name_en": get_ml_value(c.get("usualNameML"), "en"),
        "case_name_de": get_ml_value(c.get("usualNameML"), "de"),
        "jurisdictionCode": c.get("jurisdictionCode"),
        "natureCode": c.get("natureCode"),
        "introductionDate": c.get("introductionDate"),
        "introductionYear": c.get("introductionYear"),
        "closeDate": c.get("closeDate"),
        "closeYear": c.get("closeYear"),
        "affairStateCode": c.get("affairStateCode"),
        "matCode": c.get("matCode"),
        "subject_matter_text": extract_ml_labels(c.get("matCodeML"), "en"),
        "procLang": c.get("procLang"),
        "ecli": c.get("ecli"),
        "logicDocId": c.get("logicDocId"),
        "joinExist": c.get("joinExist"),
        "joinAffairs": json.dumps(c.get("joinAffairs"), ensure_ascii=False) if c.get("joinAffairs") is not None else None,
        "raw_hit_type": hit.get("type"),
        "raw_content_json": json.dumps(c, ensure_ascii=False, sort_keys=True),
    }
    return row

In [21]:
# ---------------------------------------------------------------------
# Date-window scraping
# ---------------------------------------------------------------------

def fetch_cases_for_date_window(start_date: str, end_date: str, pass_cfg: dict, depth=0):
    kwargs = {
        "filter_field": pass_cfg["filter_field"],
        "sort_term": pass_cfg["sort_term"],
        "sort_direction": pass_cfg["sort_direction"],
    }
    pass_name = pass_cfg["pass_name"]
    first_page = fetch_date_page(page_number=0, page_size=PAGE_SIZE, start_date=start_date, end_date=end_date, **kwargs)
    total_hits = int(first_page.get("totalHits", 0) or 0)
    days = window_days(start_date, end_date)

    logger.info("%s | %s to %s | hits=%s | days=%s | depth=%s", pass_name, start_date, end_date, total_hits, days, depth)
    print(f"{pass_name}: {start_date} to {end_date} | hits={total_hits:,} | days={days} | depth={depth}", flush=True)

    if total_hits == 0:
        polite_sleep(SLEEP_BETWEEN_WINDOWS_SECONDS)
        return [], []

    # Split before deep pagination becomes a problem.
    if (total_hits > MAX_HITS_PER_WINDOW or total_hits >= DEEP_PAGINATION_WARNING_AT) and days > MIN_WINDOW_DAYS:
        split = split_date_window(start_date, end_date)
        if split is not None:
            left, right = split
            left_rows, left_log = fetch_cases_for_date_window(left[0], left[1], pass_cfg, depth=depth + 1)
            right_rows, right_log = fetch_cases_for_date_window(right[0], right[1], pass_cfg, depth=depth + 1)
            return left_rows + right_rows, left_log + right_log

    n_pages = (total_hits + PAGE_SIZE - 1) // PAGE_SIZE
    rows = []
    page_log = []

    if total_hits >= DEEP_PAGINATION_WARNING_AT:
        logger.warning("%s %s-%s has %s hits and may hit deep pagination.", pass_name, start_date, end_date, total_hits)

    for page_number in range(n_pages):
        requested_from = page_number * PAGE_SIZE + 1
        requested_to = requested_from + PAGE_SIZE - 1
        if requested_from > DEEP_PAGINATION_LIMIT:
            logger.warning("Stopping %s %s-%s before deep pagination from=%s", pass_name, start_date, end_date, requested_from)
            break

        if page_number == 0:
            data = first_page
        else:
            data = fetch_date_page(page_number=page_number, page_size=PAGE_SIZE, start_date=start_date, end_date=end_date, **kwargs)

        hits = data.get("searchHits", []) or []
        kept = 0
        for i, hit in enumerate(hits):
            row = flatten_case_hit(hit, scrape_pass=pass_name, scrape_kind="date", scrape_window_start=start_date, scrape_window_end=end_date, page_number=page_number, position_on_page=i, sort_direction=pass_cfg["sort_direction"])
            if is_relevant_prefix(row):
                rows.append(row)
                kept += 1

        page_log.append({
            "scrape_pass": pass_name,
            "scrape_kind": "date",
            "window_start": start_date,
            "window_end": end_date,
            "page_number": page_number,
            "requested_from": requested_from,
            "requested_to": requested_to,
            "totalHits_reported": int(data.get("totalHits", 0) or 0),
            "hits_returned": len(hits),
            "rows_kept_prefix_CT": kept,
        })
        polite_sleep(SLEEP_BETWEEN_PAGES_SECONDS)

    polite_sleep(SLEEP_BETWEEN_WINDOWS_SECONDS)
    return rows, page_log


def run_date_pass(pass_cfg: dict):
    all_rows = []
    all_page_log = []
    for year in tqdm(range(START_YEAR, END_YEAR_EXCLUSIVE), desc=f"Scraping {pass_cfg['pass_name']}"):
        start_date = f"{year}-01-01"
        end_date = f"{year + 1}-01-01"
        try:
            rows, page_log = fetch_cases_for_date_window(start_date, end_date, pass_cfg)
            all_rows.extend(rows)
            all_page_log.extend(page_log)
        except Exception as exc:
            logger.exception("Failed %s year %s: %s", pass_cfg["pass_name"], year, exc)
            print(f"FAILED {pass_cfg['pass_name']} {year}: {exc}", flush=True)
            all_page_log.append({"scrape_pass": pass_cfg["pass_name"], "scrape_kind": "date", "window_start": start_date, "window_end": end_date, "error": repr(exc)})
    return all_rows, all_page_log

In [22]:
# ---------------------------------------------------------------------
# Prefix/year wildcard scraping: C-*/YY and T-*/YY
# ---------------------------------------------------------------------

def run_wildcard_bucket(prefix: str, year: int, pass_cfg: dict):
    search_term = bucket_search_term(prefix, year)
    pass_name = pass_cfg["pass_name"]
    sort_direction = pass_cfg["sort_direction"]

    rows = []
    page_log = []

    first_page = fetch_wildcard_page(search_term=search_term, page_number=0, page_size=PAGE_SIZE, sort_direction=sort_direction)
    total_hits = int(first_page.get("totalHits", 0) or 0)
    n_pages = (total_hits + PAGE_SIZE - 1) // PAGE_SIZE

    logger.info("%s | bucket=%s | totalHits=%s | planned pages=%s", pass_name, search_term, total_hits, n_pages)

    if total_hits >= DEEP_PAGINATION_WARNING_AT:
        logger.warning("Bucket %s/%s has %s hits; this may hit deep pagination.", pass_name, search_term, total_hits)
        print(f"WARNING: {pass_name} {search_term} has {total_hits:,} hits; may hit deep pagination.")

    consecutive_empty = 0
    seen_page_fingerprints = set()

    for page_number in range(n_pages):
        requested_from = page_number * PAGE_SIZE + 1
        requested_to = requested_from + PAGE_SIZE - 1
        if requested_from > DEEP_PAGINATION_LIMIT:
            logger.warning("Stopping bucket %s %s before deep pagination from=%s", pass_name, search_term, requested_from)
            break

        if page_number == 0:
            data = first_page
        else:
            data = fetch_wildcard_page(search_term=search_term, page_number=page_number, page_size=PAGE_SIZE, sort_direction=sort_direction)

        if SAVE_RAW_JSON:
            raw_path = COURT_INFOCURIA_JSON_DIR / f"{pass_name}_{prefix}_{year}_page_{page_number:05d}.json"
            raw_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

        hits = data.get("searchHits", []) or []
        page_ids = []
        kept = 0
        off_bucket = 0

        for i, hit in enumerate(hits):
            c = hit.get("content", hit)
            page_ids.append(str(first_nonempty(c.get("affId"), c.get("publishedId"), c.get("procedureId"), i)))
            row = flatten_case_hit(hit, scrape_pass=pass_name, scrape_kind="wildcard", bucket_prefix=prefix, bucket_year=year, bucket_search_term=search_term, page_number=page_number, position_on_page=i, sort_direction=sort_direction)
            expected_year = year
            expected_prefix = prefix
            matches_bucket = (row.get("case_prefix") == expected_prefix and row.get("case_year_full") == expected_year)
            row["matches_bucket_prefix_year"] = matches_bucket
            if not matches_bucket:
                off_bucket += 1
            if is_relevant_prefix(row):
                rows.append(row)
                kept += 1

        fingerprint = tuple(page_ids)
        repeated_page = fingerprint in seen_page_fingerprints and bool(fingerprint)
        seen_page_fingerprints.add(fingerprint)

        page_log.append({
            "scrape_pass": pass_name,
            "scrape_kind": "wildcard",
            "bucket_prefix": prefix,
            "bucket_year": year,
            "bucket_year_yy": yy_from_year(year),
            "bucket_search_term": search_term,
            "page_number": page_number,
            "requested_from": requested_from,
            "requested_to": requested_to,
            "totalHits_reported": int(data.get("totalHits", 0) or 0),
            "hits_returned": len(hits),
            "rows_kept_prefix_CT": kept,
            "off_bucket_rows_on_page": off_bucket,
            "first_id_on_page": page_ids[0] if page_ids else None,
            "last_id_on_page": page_ids[-1] if page_ids else None,
            "repeated_page_fingerprint": repeated_page,
        })

        if len(hits) == 0:
            consecutive_empty += 1
        else:
            consecutive_empty = 0

        if repeated_page:
            logger.warning("Repeated page fingerprint in %s %s page %s. Stopping bucket.", pass_name, search_term, page_number)
            break

        if consecutive_empty >= STOP_AFTER_CONSECUTIVE_EMPTY_PAGES:
            logger.warning("%s consecutive empty pages in %s %s page %s. Stopping bucket.", consecutive_empty, pass_name, search_term, page_number)
            break

        polite_sleep(SLEEP_BETWEEN_PAGES_SECONDS)

    polite_sleep(SLEEP_BETWEEN_BUCKETS_SECONDS)
    return rows, page_log


def run_wildcard_pass(pass_cfg: dict):
    all_rows = []
    all_page_log = []
    buckets = [(prefix, year) for prefix in WILDCARD_PREFIXES for year in range(START_YEAR, END_YEAR + 1)]
    print(f"{pass_cfg['pass_name']} planned buckets: {len(buckets):,}")

    for idx, (prefix, year) in enumerate(tqdm(buckets, desc=f"Scraping {pass_cfg['pass_name']}"), start=1):
        search_term = bucket_search_term(prefix, year)
        try:
            rows, page_log = run_wildcard_bucket(prefix, year, pass_cfg)
            all_rows.extend(rows)
            all_page_log.extend(page_log)
        except Exception as exc:
            logger.exception("Wildcard bucket failed %s %s: %s", pass_cfg["pass_name"], search_term, exc)
            print(f"FAILED {pass_cfg['pass_name']} {search_term}: {exc}", flush=True)
            all_page_log.append({"scrape_pass": pass_cfg["pass_name"], "scrape_kind": "wildcard", "bucket_prefix": prefix, "bucket_year": year, "bucket_search_term": search_term, "error": repr(exc)})

        if idx % CHECKPOINT_EVERY_PASS_OR_BUCKETS == 0:
            print(f"{pass_cfg['pass_name']} checkpoint bucket {idx}/{len(buckets)}: raw kept rows so far={len(all_rows):,}")
    return all_rows, all_page_log

In [23]:
# ---------------------------------------------------------------------
# Deduplication and manifest construction
# ---------------------------------------------------------------------

SLIM_MANIFEST_COLUMNS = [
    "case_number_raw",
    "case_number_clean",
    "case_prefix",
    "case_suffix",
    "case_number_int",
    "case_number_padded",
    "case_year_raw",
    "case_year_full",
    "internal_key",
    "affId",
    "procedureId",
    "case_name_en",
    "introductionDate",
    "closeDate",
    "affairStateCode",
    "joinExist",
    "joinAffairs",
]

AUDIT_COLUMN_ORDER = [
    "case_number_raw", "case_number_clean", "case_prefix", "case_suffix", "case_number_int", "case_number_padded",
    "case_year_raw", "case_year_full", "internal_key", "case_number_clean_status",
    "affId", "publishedId", "publishedAffId", "procedureId", "case_name_en", "case_name_de",
    "jurisdictionCode", "natureCode", "introductionDate", "introductionYear", "closeDate", "closeYear", "affairStateCode",
    "matCode", "subject_matter_text", "procLang", "ecli", "logicDocId", "joinExist", "joinAffairs",
    "found_in_passes", "duplicate_rows_before_dedup", "raw_case_numbers_seen", "clean_case_numbers_seen", "dedup_key", "raw_content_json",
    "scrape_pass", "scrape_kind", "scrape_window_start", "scrape_window_end",
    "bucket_prefix", "bucket_year", "bucket_year_yy", "bucket_search_term",
    "scrape_sort_direction", "scrape_page_number", "scrape_position_on_page", "raw_hit_type", "matches_bucket_prefix_year",
]


def choose_dedup_key(row):
    """
    Choose the identifier used for final deduplication.

    The raw InfoCuria affair search can return multiple procedure-level
    records for the same affId. Therefore procedureId is the primary key.

    Fallback identifiers are used only when procedureId is missing.
    """
    for col in [
        "procedureId"
    ]:
        value = row.get(col)

        if pd.notna(value) and str(value).strip():
            return f"{col}:{str(value).strip()}"

    return None


def deduplicate_cases(df_raw: pd.DataFrame) -> pd.DataFrame:
    if df_raw.empty:
        return df_raw.copy()

    df = add_case_key_columns(df_raw)
    df["dedup_key"] = df.apply(choose_dedup_key, axis=1)

    keyed = df[df["dedup_key"].notna()].copy()
    unkeyed = df[df["dedup_key"].isna()].drop_duplicates().copy()

    if keyed.empty:
        return unkeyed.reset_index(drop=True)

    pass_order = {cfg["pass_name"]: i for i, cfg in enumerate(SCRAPE_PASSES)}
    keyed["_pass_order"] = keyed["scrape_pass"].map(pass_order).fillna(999) if "scrape_pass" in keyed.columns else 999
    keyed["_source_order"] = keyed["raw_source_priority"] if "raw_source_priority" in keyed.columns else 999
    keyed = keyed.sort_values(
        [c for c in ["dedup_key", "_source_order", "_pass_order", "scrape_window_start", "bucket_year", "scrape_page_number"] if c in keyed.columns],
        na_position="last",
    )

    provenance = (
        keyed.groupby("dedup_key", dropna=False)
        .agg(
            found_in_passes=("scrape_pass", lambda x: "|".join(sorted(set(str(v) for v in x.dropna()))) if "scrape_pass" in keyed.columns else ""),
            duplicate_rows_before_dedup=("dedup_key", "size"),
            raw_case_numbers_seen=("case_number_raw", lambda x: "|".join(sorted(set(str(v) for v in x.dropna())))),
            clean_case_numbers_seen=("case_number_clean", lambda x: "|".join(sorted(set(str(v) for v in x.dropna())))),
        )
        .reset_index()
    )

    first_rows = keyed.drop_duplicates(subset=["dedup_key"], keep="first").drop(columns=["_pass_order", "_source_order"], errors="ignore")
    out = first_rows.merge(provenance, on="dedup_key", how="left")

    if not unkeyed.empty:
        out = pd.concat([out, unkeyed], ignore_index=True, sort=False)

    out = add_case_key_columns(out)
    existing = [c for c in AUDIT_COLUMN_ORDER if c in out.columns]
    extra = [c for c in out.columns if c not in existing]
    return out[existing + extra].reset_index(drop=True)


def make_slim_manifest(df_audit: pd.DataFrame) -> pd.DataFrame:
    df_audit = add_case_key_columns(df_audit)
    missing = [c for c in SLIM_MANIFEST_COLUMNS if c not in df_audit.columns]
    if missing:
        raise KeyError(f"Missing expected slim manifest columns: {missing}")
    return df_audit[SLIM_MANIFEST_COLUMNS].copy()


def pass_diagnostics(df_raw: pd.DataFrame) -> pd.DataFrame:
    if df_raw.empty:
        return pd.DataFrame()
    df_raw = add_case_key_columns(df_raw)
    group_col = "scrape_pass" if "scrape_pass" in df_raw.columns else "raw_source"
    diag = (
        df_raw.groupby(group_col, dropna=False)
        .agg(
            rows_raw=("case_number_raw", "size"),
            distinct_raw_case_numbers=("case_number_raw", "nunique"),
            distinct_clean_case_numbers=("case_number_clean", "nunique"),
            rows_without_clean_case_number=("case_number_clean", lambda x: x.isna().sum()),
            distinct_affId=("affId", "nunique"),
            distinct_internal_key=("internal_key", "nunique"),
        )
        .reset_index()
    )
    if group_col == "scrape_pass":
        order = {cfg["pass_name"]: i for i, cfg in enumerate(SCRAPE_PASSES)}
        diag["pass_order"] = diag[group_col].map(order)
        diag = diag.sort_values("pass_order").drop(columns=["pass_order"])
    return diag.reset_index(drop=True)


def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_json(path, orient="records", lines=True, dtype=False)


def write_jsonl(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_json(path, orient="records", lines=True, force_ascii=False)


def write_excel_safely(df: pd.DataFrame, path: Path, sheet_name="data"):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_excel(path, index=False, sheet_name=sheet_name)
    except Exception as exc:
        print(f"WARNING: could not write Excel file {path}: {exc}")


def save_outputs(df_raw: pd.DataFrame):
    df_raw = add_case_key_columns(df_raw)
    df_audit = deduplicate_cases(df_raw)
    df_slim = make_slim_manifest(df_audit)

    if WRITE_CSV:
        df_slim.to_csv(SCRAPE_MANIFEST_CSV, index=False, encoding="utf-8-sig", lineterminator="\n")
        df_audit.to_csv(AUDIT_MANIFEST_CSV, index=False, encoding="utf-8-sig", lineterminator="\n")

    if WRITE_JSONL:
        write_jsonl(df_raw, RAW_CASES_JSONL)
        write_jsonl(df_slim, SCRAPE_MANIFEST_JSONL)
        write_jsonl(df_audit, AUDIT_MANIFEST_JSONL)

    if WRITE_EXCEL:
        write_excel_safely(df_slim, SCRAPE_MANIFEST_XLSX, sheet_name="case_manifest")
        write_excel_safely(df_audit, AUDIT_MANIFEST_XLSX, sheet_name="case_audit")

    return df_slim, df_audit


In [24]:
# ---------------------------------------------------------------------
# Main build
# ---------------------------------------------------------------------


def scrape_all_passes() -> pd.DataFrame:
    all_rows = []

    for pass_cfg in SCRAPE_PASSES:
        print("\n" + "=" * 90)
        print(f"RUNNING PASS: {pass_cfg['pass_name']}")
        print("=" * 90)
        pass_start = datetime.now()

        if pass_cfg["kind"] == "date":
            rows, _page_log = run_date_pass(pass_cfg)
        elif pass_cfg["kind"] == "wildcard":
            rows, _page_log = run_wildcard_pass(pass_cfg)
        else:
            raise ValueError(f"Unknown pass kind: {pass_cfg['kind']}")

        for row in rows:
            row["raw_source"] = "fresh_scrape"
            row["raw_source_priority"] = 0
            row["raw_loaded_at"] = datetime.now().isoformat(timespec="seconds")

        all_rows.extend(rows)

        df_checkpoint_raw = pd.DataFrame(all_rows)
        df_checkpoint_slim, df_checkpoint_audit = save_outputs(df_checkpoint_raw)
        elapsed = datetime.now() - pass_start
        print(
            f"PASS DONE: {pass_cfg['pass_name']} | "
            f"pass rows kept={len(rows):,} | "
            f"cumulative raw rows={len(all_rows):,} | "
            f"cumulative dedup rows={len(df_checkpoint_slim):,} | "
            f"elapsed={elapsed}"
        )

    return pd.DataFrame(all_rows)


def load_cached_raw_cases() -> pd.DataFrame:
    """
    Load existing case data without scraping.

    v3 fallback order:
      1. raw JSONL cache in output/court_infocuria/jsonl/
      2. audit Excel manifest in output/court_infocuria/excel/
      3. slim Excel manifest in output/court_infocuria/excel/
      4. legacy/root audit CSV in output/court_infocuria/
      5. legacy/root slim CSV in output/court_infocuria/

    The Excel fallbacks are intentionally preferred over root CSV fallbacks because
    manual inspection/editing usually happens in output/court_infocuria/excel/.
    """
    if RAW_CASES_JSONL.exists():
        print("Loading cached raw scraped cases JSONL:", RAW_CASES_JSONL)
        df = read_jsonl(RAW_CASES_JSONL)
        if not df.empty:
            if "raw_source" not in df.columns:
                df["raw_source"] = "cached_jsonl"
            df["raw_source_priority"] = 1
        return df

    if AUDIT_MANIFEST_XLSX.exists():
        print("Raw JSONL not found. Falling back to audit Excel:", AUDIT_MANIFEST_XLSX)
        df = pd.read_excel(AUDIT_MANIFEST_XLSX, dtype=str)
        if not df.empty:
            df["raw_source"] = "fallback_audit_excel"
            df["raw_source_priority"] = 2
        return df

    if SCRAPE_MANIFEST_XLSX.exists():
        print("Raw JSONL and audit Excel not found. Falling back to slim Excel:", SCRAPE_MANIFEST_XLSX)
        df = pd.read_excel(SCRAPE_MANIFEST_XLSX, dtype=str)
        if not df.empty:
            df["raw_source"] = "fallback_slim_excel"
            df["raw_source_priority"] = 3
        return df

    if AUDIT_MANIFEST_CSV.exists():
        print("Raw JSONL/Excel not found. Falling back to legacy/root audit CSV:", AUDIT_MANIFEST_CSV)
        df = pd.read_csv(AUDIT_MANIFEST_CSV, dtype=str, low_memory=False)
        if not df.empty:
            df["raw_source"] = "fallback_audit_csv"
            df["raw_source_priority"] = 4
        return df

    if SCRAPE_MANIFEST_CSV.exists():
        print("Raw JSONL/Excel/audit CSV not found. Falling back to legacy/root slim CSV:", SCRAPE_MANIFEST_CSV)
        df = pd.read_csv(SCRAPE_MANIFEST_CSV, dtype=str, low_memory=False)
        if not df.empty:
            df["raw_source"] = "fallback_slim_csv"
            df["raw_source_priority"] = 5
        return df

    raise FileNotFoundError(
        "No cached case data found. Use SCRAPE_MODE = 'fresh_scrape' first, or provide existing JSONL/Excel/CSV manifests."
    )


def build_or_load_manifest():
    if SCRAPE_MODE == "fresh_scrape":
        print("Mode: fresh_scrape — scraping from zero and overwriting outputs.")
        df_raw = scrape_all_passes()
        df_slim, df_audit = save_outputs(df_raw)
        return df_slim, df_audit, df_raw

    if SCRAPE_MODE == "update_scrape":
        print("Mode: update_scrape — loading cached raw data, scraping again, then merging/updating outputs.")
        df_cached = load_cached_raw_cases()
        if not df_cached.empty:
            df_cached["raw_source_priority"] = 1
        df_new = scrape_all_passes()
        if not df_new.empty:
            df_new["raw_source_priority"] = 0
        df_raw = pd.concat([df_new, df_cached], ignore_index=True, sort=False)
        df_raw = add_case_key_columns(df_raw)
        identity_cols = [c for c in ["affId", "publishedId", "publishedAffId", "procedureId", "case_number_raw", "scrape_pass"] if c in df_raw.columns]
        if identity_cols:
            df_raw = df_raw.sort_values("raw_source_priority", na_position="last").drop_duplicates(subset=identity_cols, keep="first")
        df_slim, df_audit = save_outputs(df_raw)
        return df_slim, df_audit, df_raw

    if SCRAPE_MODE == "rebuild_from_cache":
        print("Mode: rebuild_from_cache — no scraping; rebuilding outputs from saved JSONL/Excel/CSV.")
        df_raw = load_cached_raw_cases()
        df_raw = add_case_key_columns(df_raw)
        df_slim, df_audit = save_outputs(df_raw)
        return df_slim, df_audit, df_raw

    raise ValueError(f"Unsupported SCRAPE_MODE: {SCRAPE_MODE}")


df_cases, df_cases_audit, df_raw_all = build_or_load_manifest()

print("\nFinal slim manifest rows after deduplication:", f"{len(df_cases):,}")
print("Final audit manifest rows after deduplication:", f"{len(df_cases_audit):,}")
print("Raw rows available before final deduplication:", f"{len(df_raw_all):,}")
print("Slim scrape manifest CSV:", SCRAPE_MANIFEST_CSV)
print("Audit scrape manifest CSV:", AUDIT_MANIFEST_CSV)
print("Raw cases JSONL:", RAW_CASES_JSONL)
print("Slim scrape manifest Excel:", SCRAPE_MANIFEST_XLSX)
print("Audit scrape manifest Excel:", AUDIT_MANIFEST_XLSX)


Mode: rebuild_from_cache — no scraping; rebuilding outputs from saved JSONL/Excel/CSV.
Loading cached raw scraped cases JSONL: /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_case_scrape_raw_cases.jsonl

Final slim manifest rows after deduplication: 56,828
Final audit manifest rows after deduplication: 56,828
Raw rows available before final deduplication: 315,746
Slim scrape manifest CSV: /home/edik/projects/eccjeu/output/court_infocuria/infocuria_case_scrape_manifest.csv
Audit scrape manifest CSV: /home/edik/projects/eccjeu/output/court_infocuria/infocuria_case_scrape_audit_manifest.csv
Raw cases JSONL: /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_case_scrape_raw_cases.jsonl
Slim scrape manifest Excel: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_case_scrape_manifest.xlsx
Audit scrape manifest Excel: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_case_scrape_audit_manifest.xlsx


In [25]:
# ---------------------------------------------------------------------
# Final checks: counts and columns
# ---------------------------------------------------------------------

print("=" * 90)
print("FINAL CHECKS")
print("=" * 90)

print("Slim manifest rows:", f"{len(df_cases):,}")
print("Audit manifest rows:", f"{len(df_cases_audit):,}")

if not df_cases.empty:
    print("Distinct raw displayed identifiers:", f"{df_cases['case_number_raw'].nunique(dropna=True):,}")
    print("Distinct clean base case numbers:", f"{df_cases['case_number_clean'].nunique(dropna=True):,}")
    print("Rows without clean case number:", f"{df_cases['case_number_clean'].isna().sum():,}")
    print("Slim manifest columns:")
    display(pd.DataFrame({"column": df_cases.columns.tolist()}))
    print("Prefix counts:")
    display(df_cases["case_prefix"].value_counts(dropna=False).reset_index(name="rows"))

print("\nPer-pass counts before final deduplication, displayed only and not written as CSV:")
diag = pass_diagnostics(df_raw_all)
display(diag)

print("\nOverlap / provenance after deduplication, from audit manifest:")
if "found_in_passes" in df_cases_audit.columns:
    display(df_cases_audit["found_in_passes"].value_counts(dropna=False).reset_index(name="dedup_rows"))

print("\nCase-year summary, tail:")
if not df_cases.empty:
    case_year_summary = (
        df_cases.groupby("case_year_full", dropna=False)
        .agg(rows=("case_number_raw", "size"), distinct_raw=("case_number_raw", "nunique"), distinct_clean=("case_number_clean", "nunique"))
        .reset_index()
        .sort_values("case_year_full", na_position="last")
    )
    display(case_year_summary.tail(20))


FINAL CHECKS
Slim manifest rows: 56,828
Audit manifest rows: 56,828
Distinct raw displayed identifiers: 53,265
Distinct clean base case numbers: 53,067
Rows without clean case number: 0
Slim manifest columns:


,column
0,case_number_raw
1,case_number_clean
2,case_prefix
3,case_suffix
4,case_number_int
5,case_number_padded
6,case_year_raw
7,case_year_full
8,internal_key
9,affId


Prefix counts:


,case_prefix,rows
0,C,30343
1,T,26485



Per-pass counts before final deduplication, displayed only and not written as CSV:


,scrape_pass,rows_raw,distinct_raw_case_numbers,distinct_clean_case_numbers,rows_without_clean_case_number,distinct_affId,distinct_internal_key
0,introductionDate_ASC,56817,53259,53061,0,51649,53061
1,introductionDate_DESC,56607,53056,52858,0,51452,52858
2,closeDate_ASC,53561,50053,49855,0,48516,49855
3,closeDate_DESC,53560,50053,49855,0,48518,49855
4,case_number_wildcard_ASC,47601,44106,44106,0,44143,44106
5,case_number_wildcard_DESC,47600,44106,44106,0,44143,44106



Overlap / provenance after deduplication, from audit manifest:

Case-year summary, tail:


,case_year_full,rows,distinct_raw,distinct_clean
54,2007,1221,1148,1148
55,2008,1359,1267,1266
56,2009,1259,1170,1170
57,2010,1422,1333,1332
58,2011,1561,1463,1463
59,2012,1328,1256,1256
60,2013,1641,1510,1510
61,2014,1798,1579,1579
62,2015,1690,1584,1584
63,2016,1844,1735,1734


In [26]:
# ---------------------------------------------------------------------
# Optional spot checks
# ---------------------------------------------------------------------

spot_checks = ["T-681/23", "T-684/23", "T-766/23", "T-336/25", "C-431/26", "C-530/26"]
if not df_cases.empty:
    mask = df_cases["case_number_clean"].isin(spot_checks)
    print("Spot-check hits:")
    cols = [c for c in [
        "case_number_raw", "case_number_clean", "case_number_padded", "affId", "procedureId",
        "introductionDate", "closeDate", "case_name_en"
    ] if c in df_cases.columns]
    display(df_cases.loc[mask, cols])


Spot-check hits:


,case_number_raw,case_number_clean,case_number_padded,affId,procedureId,introductionDate,closeDate,case_name_en
22604,C-431/26,C-431/26,0431,C/0431/26/00000000RP/01,C/0431/26/00000000RP/01/P/01,2026-04-28,NaN,Associação Ius Omnibus III
25468,C-530/26,C-530/26,0530,C/0530/26/00000000RP/01,C/0530/26/00000000RP/01/P/01,2026-05-21,NaN,ANAV
45175,T-336/25,T-336/25,0336,T/0336/25/00000000RD/01,T/0336/25/00000000RD/01/P/01,2025-05-27,NaN,Nkubito v Council
54254,T-681/23,T-681/23,0681,T/0681/23/00000000RD/01,T/0681/23/00000000RD/01/P/01,2023-10-08,2025-12-17,ZZ v Parliament
54300,T-684/23,T-684/23,0684,T/0684/23/00000000RD/01,T/0684/23/00000000RD/01/P/01,2023-10-08,2025-12-17,ZZ v Parliament
55473,T-766/23,T-766/23,0766,T/0766/23/00000000RD/01,T/0766/23/00000000RD/01/P/01,2023-10-08,2025-12-17,ZZ v Parliament


In [27]:
print("Done.")
print("SCRAPE_MODE:", SCRAPE_MODE)
print("Slim scrape manifest CSV:", SCRAPE_MANIFEST_CSV)
print("Audit scrape manifest CSV:", AUDIT_MANIFEST_CSV)
print("Raw cases JSONL:", RAW_CASES_JSONL)
print("Slim scrape manifest JSONL:", SCRAPE_MANIFEST_JSONL)
print("Audit scrape manifest JSONL:", AUDIT_MANIFEST_JSONL)
print("Slim scrape manifest Excel:", SCRAPE_MANIFEST_XLSX)
print("Audit scrape manifest Excel:", AUDIT_MANIFEST_XLSX)
print("Rows slim:", len(df_cases))
print("Rows audit:", len(df_cases_audit))
print("Distinct clean cases slim:", df_cases["case_number_clean"].nunique(dropna=True) if not df_cases.empty else 0)
print("Distinct internal keys slim:", df_cases["internal_key"].nunique(dropna=True) if "internal_key" in df_cases.columns and not df_cases.empty else 0)


Done.
SCRAPE_MODE: rebuild_from_cache
Slim scrape manifest CSV: /home/edik/projects/eccjeu/output/court_infocuria/infocuria_case_scrape_manifest.csv
Audit scrape manifest CSV: /home/edik/projects/eccjeu/output/court_infocuria/infocuria_case_scrape_audit_manifest.csv
Raw cases JSONL: /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_case_scrape_raw_cases.jsonl
Slim scrape manifest JSONL: /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_case_scrape_manifest.jsonl
Audit scrape manifest JSONL: /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_case_scrape_audit_manifest.jsonl
Slim scrape manifest Excel: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_case_scrape_manifest.xlsx
Audit scrape manifest Excel: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_case_scrape_audit_manifest.xlsx
Rows slim: 56828
Rows audit: 56828
Distinct clean cases slim: 53067
Distinct internal keys slim: 53067
